# 01 — Filters & Feature Maps: What Has Each Model Learned?

**Goal of this notebook**: directly *see* the difference transfer learning makes,
rather than just compare accuracy numbers. Two views:

1. **Filters** (learned weights) — what visual pattern does each early filter respond
   to, regardless of any particular input image?
2. **Feature maps** (activations) — for one specific MRI, what does each layer
   actually produce?

**Intent of comparing custom CNN vs DenseNet here specifically**: the custom CNN's
first-layer filters start as random noise and only become meaningful after training
on ~2,400 images. DenseNet's first-layer filters were already trained on 1.4M
ImageNet images — they encode general edge/color/texture detectors before your MRI
data ever touches them. Seeing both side by side is the clearest possible illustration
of why transfer learning outperforms training from scratch on a small dataset.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from src.interpretability import get_feature_maps, get_first_layer_filters

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)

## Load trained models

**Intent**: we want the *trained* custom CNN (not a freshly-initialized one) to see
what its filters became after training — compare that against DenseNet's filters,
which are meaningful even before any of your training happens.

In [ ]:
custom_cnn = tf.keras.models.load_model("../models/saved_models/custom_cnn_baseline_best.keras")
densenet_model = tf.keras.models.load_model(
    "../models/saved_models/densenet121_phase2_finetuned_best.keras"
)
densenet_base = densenet_model.get_layer("densenet121")

## First-layer filters, side by side

**Intent**: visualize the actual learned kernels. Custom CNN filters (trained on MRI
only) vs DenseNet filters (pretrained on ImageNet, then fine-tuned on MRI) — look for
whether the custom CNN's filters resemble recognizable edge/blob detectors, or still
look close to noise given the limited training data.

In [ ]:
def plot_filters(filters, title, n_filters=16):
    n_filters = min(n_filters, filters.shape[-1])
    fig, axes = plt.subplots(2, n_filters // 2, figsize=(14, 4))
    for i, ax in enumerate(axes.flat):
        f = filters[:, :, :, i]
        f_normalized = (f - f.min()) / (f.max() - f.min() + 1e-8)
        if f_normalized.shape[-1] == 1:
            ax.imshow(f_normalized.squeeze(), cmap="gray")
        else:
            ax.imshow(f_normalized)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


custom_filters = get_first_layer_filters(custom_cnn)
plot_filters(custom_filters, "Custom CNN — first-layer filters (trained on ~2.4k MRI slices)")

densenet_filters = get_first_layer_filters(densenet_base)
plot_filters(
    densenet_filters, "DenseNet121 — first-layer filters (pretrained on 1.4M ImageNet images)"
)

## Feature maps for one real MRI slice

**Intent**: filters show *general* learned patterns; feature maps show what a
*specific* image activates. Pick one test-set image and look at what each model's
deeper layers respond to — do the activations concentrate around the tumor region,
or scatter across the whole image (a sign the model hasn't learned to localize
anything useful)?

In [ ]:
from src.data_utils import load_image_as_array

split_metadata = pd.read_csv("../data/processed/metadata_split.csv")
sample_row = split_metadata[split_metadata["split"] == "test"].sample(1, random_state=1).iloc[0]

sample_image = load_image_as_array(sample_row["image_path"], sample_row["source_dataset"], IMG_SIZE)

print(f"Sample: true label = {sample_row['label']}")
plt.imshow(sample_image.squeeze(), cmap="gray")
plt.title(f"Sample MRI — true label: {sample_row['label']}")
plt.axis("off")
plt.show()

In [ ]:
from src.interpretability import find_last_conv_layer_name

last_conv_custom = find_last_conv_layer_name(custom_cnn)
feature_map = get_feature_maps(custom_cnn, sample_image, last_conv_custom)

n_channels_to_show = min(16, feature_map.shape[-1])
fig, axes = plt.subplots(2, n_channels_to_show // 2, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_map[:, :, i], cmap="viridis")
    ax.axis("off")
fig.suptitle(f"Custom CNN — feature maps at '{last_conv_custom}' for this sample")
plt.tight_layout()
plt.show()

## Conclusion

Fill in after running: describe what you actually see — do DenseNet's filters look
like recognizable edge/texture detectors even before fine-tuning, while the custom
CNN's filters look noisier? That visual difference *is* the reason transfer learning
outperforms the baseline on this dataset size — not a training-time hyperparameter,
but a fundamentally different starting point for what the network already "knows."

Next notebook: **02_gradcam_interpretability.ipynb** — see *where* each model looks
when it makes a prediction, not just what patterns it has learned in general.